## 1) Environment Setup
Mount Google Drive and install required libraries.

In [ ]:
!nvidia-smi

Mon Apr  6 06:20:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   32C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
!pip install -q transformers datasets evaluate rouge_score

## 2) Data Extraction and Sampling
Extract raw data, refresh destination folders, and build train/dev/test samples.

In [ ]:
import os

file_path = '/content/drive/MyDrive/Baginda/liputan6_data.tar.gz'
local_extract_path = '/content/temp_extract'

# Create local directory
os.makedirs(local_extract_path, exist_ok=True)

print(f"Extracting 'canonical' folder to local storage {local_extract_path}...")
# Extract only the canonical directory from the archive
!tar -xzf "{file_path}" -C "{local_extract_path}" liputan6_data/canonical
print("Local extraction finished.")

In [ ]:
import shutil
import os

sampled_extract_path = '/content/sampled_data'

print(f"Deleting old sampled folder at {sampled_extract_path}...")
if os.path.exists(sampled_extract_path):
    shutil.rmtree(sampled_extract_path)
    print("Old folder successfully deleted.")
else:
    print("Old folder does not exist.")

# Recreate the base directory
os.makedirs(os.path.join(sampled_extract_path, 'canonical'), exist_ok=True)
print(f"Created fresh directory at {os.path.join(sampled_extract_path, 'canonical')}")


In [ ]:
import os
import random
import shutil

# Configuration
TOTAL_FILES_TO_SAMPLE = 25000
train_ratio = 0.70
dev_ratio = 0.10
test_ratio = 0.20

# Paths
local_base_path = '/content/temp_extract/liputan6_data/canonical'
sampled_base_path = '/content/sampled_data/canonical'

# Calculate counts
train_count = int(TOTAL_FILES_TO_SAMPLE * train_ratio)
dev_count = int(TOTAL_FILES_TO_SAMPLE * dev_ratio)
test_count = TOTAL_FILES_TO_SAMPLE - train_count - dev_count # Ensure it adds up exactly

splits = {
    'train': train_count,
    'dev': dev_count,
    'test': test_count
}

print(f"Target distribution -> Train: {train_count}, Dev: {dev_count}, Test: {test_count}\n")

for split, count in splits.items():
    src_dir = os.path.join(local_base_path, split)
    dst_dir = os.path.join(sampled_base_path, split)

    os.makedirs(dst_dir, exist_ok=True)

    if os.path.exists(src_dir):
        # Get all json files in the source directory
        all_files = [f for f in os.listdir(src_dir) if f.endswith('.json')]
        print(f"Found {len(all_files)} files in local {split} folder.")

        # Sample files (or take all if count > available files)
        files_to_copy = random.sample(all_files, min(count, len(all_files)))

        print(f"Copying {len(files_to_copy)} files to local sampled {split} folder...")
        for f in files_to_copy:
            shutil.copy2(os.path.join(src_dir, f), os.path.join(dst_dir, f))
        print(f"Finished copying {split}.\n")
    else:
        print(f"Warning: Source directory {src_dir} not found!")

print("Sampling and copying process completed!")


## 3) Data Sanity Check
Inspect one sample file to verify schema and content shape.

In [ ]:
import os
import json

sample_dir = '/content/sampled_data/canonical/train'

if os.path.exists(sample_dir):
    sample_files = [f for f in os.listdir(sample_dir) if f.endswith('.json')]
    if sample_files:
        sample_file_path = os.path.join(sample_dir, sample_files[0])
        print(f"Reading sample file: {sample_file_path}\n")

        with open(sample_file_path, 'r') as f:
            data = json.load(f)

        print("JSON Keys:", list(data.keys()))
        print("\nFull JSON Content:")
        print(json.dumps(data, indent=2))
    else:
        print("No JSON files found in the directory.")
else:
    print(f"Directory not found: {sample_dir}")


## 4) Load and Structure Dataset
Load JSON files into pandas DataFrames for each split.

In [ ]:
import os
import json
import pandas as pd
from tqdm.notebook import tqdm

def load_data_split(split_path):
    records = []
    if not os.path.exists(split_path):
        print(f"Path not found: {split_path}")
        return pd.DataFrame()

    files = [f for f in os.listdir(split_path) if f.endswith('.json')]
    for f in tqdm(files, desc=f"Loading {os.path.basename(split_path)}"):
        with open(os.path.join(split_path, f), 'r') as file:
            data = json.load(file)

            # Form the full text from nested lists
            article = " ".join([" ".join(sentence) for sentence in data.get('clean_article', [])])
            summary = " ".join([" ".join(sentence) for sentence in data.get('clean_summary', [])])

            records.append({
                'id': data.get('id'),
                'url': data.get('url'),
                'article': article,
                'summary': summary,
                'extractive_summary': data.get('extractive_summary')
            })

    return pd.DataFrame(records)

# Base directory where your sampled data is stored locally
base_dir = '/content/sampled_data/canonical'

# Load the datasets
print("Loading datasets into Pandas DataFrames...")
df_train = load_data_split(os.path.join(base_dir, 'train'))
df_dev = load_data_split(os.path.join(base_dir, 'dev'))
df_test = load_data_split(os.path.join(base_dir, 'test'))

print("\nData Loading Complete!")
print(f"Train size: {len(df_train)}")
print(f"Dev size: {len(df_dev)}")
print(f"Test size: {len(df_test)}")

# Preview the train dataframe
display(df_train.head())


## 5) Tokenization and Preprocessing
Convert DataFrames to Hugging Face datasets and tokenize inputs/targets.

In [ ]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

# Convert Pandas DataFrames to Hugging Face Datasets
raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(df_train),
    "validation": Dataset.from_pandas(df_dev),
    "test": Dataset.from_pandas(df_test)
})

# We use a BERT2BERT model tailored for Indonesian text summarization
model_checkpoint = "cahya/bert2bert-indonesian-summarization"

print(f"Loading tokenizer from {model_checkpoint}...")
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Set max lengths for the articles and the summaries
max_input_length = 512
max_target_length = 128

def preprocess_function(examples):
    inputs = examples["article"]
    targets = examples["summary"]

    # Tokenize the input articles
    model_inputs = tokenizer(inputs, max_length=max_input_length, padding="max_length", truncation=True)

    # Tokenize the target summaries
    labels = tokenizer(targets, max_length=max_target_length, padding="max_length", truncation=True)

    # Replace the padding token id with -100 so they are ignored by the loss function
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing datasets... This might take a minute.")
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)
print("\nTokenization complete!")

# Preview the keys of the first tokenized train record to ensure it contains 'input_ids' and 'labels'
print("\nFeatures in our tokenized dataset:", list(tokenized_datasets["train"][0].keys()))

## 6) Evaluate ROUGE-1, ROUGE-2, AND ROUGE-L scores.
Assuming that the datasets provided is trained by indoBERT. We Evaluate the ROUGE scores from the train datasets, which conclude its low accuracy for indonesian text summarization based on the paper in the pdf file provided in the project.

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

scores = scorer.score(
    target="Liputan6 . com , Jakarta : Artis Ussy Sulistiawaty mengaku sedikit mengerem kegiatan di bulan suci Ramadan . Tujuannya demi menjaga kondisi tubuh yang belakangan rentan terhadap penyakit . \" Rada dikurangi saja , sekarang lebih fokus sama rekaman buat album ketiga , \"ujar Ussy di Hot Shot SCTV , Ahad ( 29/8 ) . Ussy saat ini berada pada kondisi 90 persen fit . Ussy yang pada saat itu ditemani sang kekasih , Andika Pratama berencana meluncurkan album terbaru setelah Lebaran . Pelantun tembang Klik itu berharap deretan lagu barunya mampu memikat hati para pecinta musik Tanah Air . Sebelumnya Ussy dan Andika berkomitmen hanya bertemu setelah berbuka puasa . Itu semua dilakukan demi menghindari hal negatif yang dapat merusak ibadah mereka [ baca : Andhika-Ussy Bicara Soal Makna Puasa ] . ( AIS ) .",
    prediction="Ussy Sulistiawaty mengaku sedikit mengerem kegiatan di bulan Ramadan untuk menjaga kondisi tubuh ."
)

print(scores)

In [ ]:
import pandas as pd
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def compute_rouge(row):
    scores = scorer.score(
        target=row['article'],
        prediction=row['summary']
    )
    result = {}
    for m in ['rouge1', 'rouge2', 'rougeL']:
        result[f'{m}_precision'] = scores[m].precision
        result[f'{m}_recall'] = scores[m].recall
        result[f'{m}_fmeasure'] = scores[m].fmeasure
    return pd.Series(result)

rouge_scores = df_train.apply(compute_rouge, axis=1)
df = pd.concat([df_train, rouge_scores], axis=1)

cols = [
    'rouge1_precision', 'rouge1_recall', 'rouge1_fmeasure',
    'rouge2_precision', 'rouge2_recall', 'rouge2_fmeasure',
    'rougeL_precision', 'rougeL_recall', 'rougeL_fmeasure'
]

print(df[cols].agg(['min', 'max', 'mean']).round(4))

## 7) Model and Trainer Setup
Initialize the seq2seq model, metrics, and training arguments.

In [ ]:
from transformers import EncoderDecoderModel, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import evaluate
import numpy as np

# 1. Load the model
print(f"Loading model {model_checkpoint}...")
model = EncoderDecoderModel.from_pretrained(model_checkpoint)
model.config.tie_word_embeddings = False

# Set model configuration for sequence-to-sequence generation
model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.eos_token_id = tokenizer.sep_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.vocab_size = model.config.encoder.vocab_size

# Set generation parameters on generation_config (required for transformers >= 4.36)
model.generation_config.max_length = 128
model.generation_config.min_length = 16
model.generation_config.no_repeat_ngram_size = 3
model.generation_config.early_stopping = True
model.generation_config.length_penalty = 2.0
model.generation_config.num_beams = 4
model.generation_config.decoder_start_token_id = tokenizer.cls_token_id
model.generation_config.eos_token_id = tokenizer.sep_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id

# 2. Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# 3. Metric (ROUGE)
rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Rouge expects a newline after each sentence
    decoded_preds = ["\n".join(pred.strip().split()) for pred in decoded_preds]
    decoded_labels = ["\n".join(label.strip().split()) for label in decoded_labels]

    result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    # Extract a few results
    result = {key: value * 100 for key, value in result.items()}

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

# 4. Training Arguments
batch_size = 8

args = Seq2SeqTrainingArguments(
    output_dir="/content/bert2bert-indonesian-summarization-finetuned",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True, # Enable mixed precision for faster training on GPU
    push_to_hub=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    # tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print("Trainer setup complete! Ready to start training.")


## 8) Pre-fine-tuning Baseline Evaluation
Generate summaries using the **pre-trained** (not yet fine-tuned) model on a 200-sample subset of the test set.
This establishes a before/after ROUGE comparison to quantify the benefit of fine-tuning.


In [ ]:
import torch
import numpy as np
from rouge_score import rouge_scorer as rs

BASELINE_SAMPLE_SIZE = 200
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Draw a reproducible 200-sample subset from the test set
baseline_sample = df_test.sample(n=BASELINE_SAMPLE_SIZE, random_state=42).reset_index(drop=True)

def generate_summary(text, mdl, tok, max_input=512, max_new=128):
    """Tokenize an article and generate an abstractive summary via beam search."""
    inputs = tok(
        text,
        max_length=max_input,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        output_ids = mdl.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_new,
            min_length=16,
            num_beams=4,
            no_repeat_ngram_size=3,
            length_penalty=2.0,
            early_stopping=True,
            decoder_start_token_id=tok.cls_token_id,
        )
    return tok.decode(output_ids[0], skip_special_tokens=True)

print("Generating pre-fine-tuning summaries on 200 test samples...")
pre_ft_generated = []
for _, row in baseline_sample.iterrows():
    pred = generate_summary(row["article"], model, tokenizer)
    pre_ft_generated.append(pred)
print(f"Done. Generated {len(pre_ft_generated)} summaries.")

# Compute ROUGE on generated summaries vs. reference human summaries
scorer_eval = rs.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

agg_pre = {"rouge1": [], "rouge2": [], "rougeL": []}
for pred, ref in zip(pre_ft_generated, baseline_sample["summary"].tolist()):
    s = scorer_eval.score(target=ref, prediction=pred)
    for k in agg_pre:
        agg_pre[k].append(s[k].fmeasure)

pre_ft_scores = {k: round(float(np.mean(v)) * 100, 4) for k, v in agg_pre.items()}
print("\n--- Pre-fine-tuning ROUGE Scores (model-generated vs. reference, 200 test samples) ---")
for metric, val in pre_ft_scores.items():
    print(f"  {metric:8s}: {val:.4f}")


## 9) Fine-tuning
Fine-tune the BERT2BERT model on the 17,500-sample Indonesian training split.
After training the model is saved back to Google Drive so it can be reloaded in future sessions.

In [ ]:
import os

# Run fine-tuning
print("Starting fine-tuning...")
train_result = trainer.train()

# Print training metrics
print("\n--- Training Metrics ---")
print(train_result.metrics)

# Save fine-tuned model and tokenizer to Google Drive (for reuse in future sessions)
finetuned_save_path = "/content/drive/MyDrive/Baginda/bert2bert-indonesian-finetuned"
os.makedirs(finetuned_save_path, exist_ok=True)
trainer.save_model(finetuned_save_path)
tokenizer.save_pretrained(finetuned_save_path)
print(f"\nFine-tuned model saved to: {finetuned_save_path}")

## 10) Post-fine-tuning Evaluation
Evaluate the fine-tuned model in two ways:

1. **Trainer evaluate** — official ROUGE on the full 5,000-sample tokenized test split (uses `compute_metrics` with `predict_with_generate=True`).
2. **Direct inference ROUGE** — run `model.generate()` on the **same 200-sample baseline subset** used in Section 8, so results are directly comparable.


In [ ]:
# 10a. Official Trainer evaluation on the full tokenized test split
print("Running evaluation on the full tokenized test split (5,000 samples)...")
eval_results = trainer.evaluate(tokenized_datasets["test"])
print("\n--- Official Test-set ROUGE (full 5,000 samples) ---")
for k, v in eval_results.items():
    print(f"  {k}: {v}")


In [ ]:
# 10b. Direct inference ROUGE on the same 200-sample baseline subset
print("Generating post-fine-tuning summaries on the same 200 baseline test samples...")
post_ft_generated = []
model.eval()
for _, row in baseline_sample.iterrows():
    pred = generate_summary(row["article"], model, tokenizer)
    post_ft_generated.append(pred)
print(f"Done. Generated {len(post_ft_generated)} summaries.")

agg_post = {"rouge1": [], "rouge2": [], "rougeL": []}
for pred, ref in zip(post_ft_generated, baseline_sample["summary"].tolist()):
    s = scorer_eval.score(target=ref, prediction=pred)
    for k in agg_post:
        agg_post[k].append(s[k].fmeasure)

post_ft_scores = {k: round(float(np.mean(v)) * 100, 4) for k, v in agg_post.items()}
print("\n--- Post-fine-tuning ROUGE Scores (model-generated vs. reference, 200 test samples) ---")
for metric, val in post_ft_scores.items():
    print(f"  {metric:8s}: {val:.4f}")


## 11) Comparison Table & Qualitative Demo

### 11a) ROUGE Comparison Table
Compare our model (before and after fine-tuning) against the paper's published baselines on the Liputan6 canonical test set.

### 11b) Qualitative Examples
Inspect 5 sample articles side-by-side: reference summary, pre-fine-tuned output, and post-fine-tuned output.


In [ ]:
import pandas as pd

# Paper-reported baselines on the canonical test set (Table 3, AACL 2020 paper)
paper_baselines = {
    "Lead-2 (paper)":              {"rouge1": 36.68, "rouge2": 20.23, "rougeL": 33.71},
    "PTGen (paper)":               {"rouge1": 36.10, "rouge2": 19.19, "rougeL": 33.56},
    "BertAbs mBERT (paper)":       {"rouge1": 39.48, "rouge2": 21.59, "rougeL": 36.72},
    "BertExtAbs mBERT (paper)":    {"rouge1": 39.81, "rouge2": 21.84, "rougeL": 37.02},
    "BertAbs IndoBERT (paper)":    {"rouge1": 40.94, "rouge2": 23.01, "rougeL": 37.89},
    "BertExtAbs IndoBERT (paper) ★": {"rouge1": 41.08, "rouge2": 22.85, "rougeL": 38.01},
}

# Our model results on the 200-sample baseline subset
our_results = {
    "BERT2BERT (pre-fine-tuning)":  {k: pre_ft_scores[k]  for k in ["rouge1", "rouge2", "rougeL"]},
    "BERT2BERT (fine-tuned ours)":  {k: post_ft_scores[k] for k in ["rouge1", "rouge2", "rougeL"]},
}

all_results = {**paper_baselines, **our_results}
comparison_df = pd.DataFrame(all_results).T.rename(columns={
    "rouge1": "ROUGE-1", "rouge2": "ROUGE-2", "rougeL": "ROUGE-L"
})
comparison_df = comparison_df.round(2)

print("=" * 60)
print("ROUGE Comparison — Liputan6 Canonical Test Set")
print("(Paper baselines: full ~10,972 test samples;")
print(" Our model: 200 randomly sampled test articles)")
print("=" * 60)
display(comparison_df)


In [ ]:
DEMO_N = 5

print("=" * 80)
print("Qualitative Demo — 5 Sample Summaries")
print("=" * 80)

for i in range(DEMO_N):
    row = baseline_sample.iloc[i]
    # Truncate article preview to first 300 chars to keep output readable
    article_preview = row["article"][:300].rsplit(" ", 1)[0] + " ..."

    print(f"\n[Sample {i+1}]")
    print(f"  Article (first 300 chars):\n    {article_preview}")
    print(f"\n  Reference Summary:\n    {row['summary']}")
    print(f"\n  Pre-fine-tuning Generated:\n    {pre_ft_generated[i]}")
    print(f"\n  Post-fine-tuning Generated:\n    {post_ft_generated[i]}")
    print("-" * 80)


# Comprehensive Notebook Summary

## 1) Environment Setup
* Mounts Google Drive to access the dataset.
* Installs required libraries for the project: `transformers`, `datasets`, `evaluate`, and `rouge_score`.
* Verifies GPU availability and status using `!nvidia-smi`.

## 2) Data Extraction and Sampling
* Extracts the `canonical` folder from the Liputan6 dataset archive (`.tar.gz`).
* Cleans up any existing extraction folders to ensure a fresh start.
* Samples a total of 25,000 files and distributes them into three splits: Train (70%, 17,500 files), Dev (10%, 2,500 files), and Test (20%, 5,000 files).

## 3) Data Sanity Check
* Inspects a single JSON sample file from the training directory.
* Verifies the dataset's schema (keys like `id`, `clean_article`, `clean_summary`) and structure to ensure correct data extraction.

## 4) Load and Structure Dataset
* Parses the sampled JSON files and loads them into Pandas DataFrames for train, dev, and test splits.
* Joins nested lists of sentences into continuous text strings for both articles and their corresponding summaries.

## 5) Tokenization and Preprocessing
* Converts the Pandas DataFrames into a Hugging Face `DatasetDict`.
* Loads the pre-trained tokenizer for `cahya/bert2bert-indonesian-summarization`.
* Applies a preprocessing function to tokenize inputs (articles, max length 512) and targets (summaries, max length 128), and replaces padding token IDs with `-100` so they are ignored by the loss function.

## 6) Evaluate ROUGE-1, ROUGE-2, AND ROUGE-L scores
* Sets up the `rouge_scorer` to calculate precision, recall, and f-measure.
* Runs a baseline evaluation on the training set to demonstrate the low accuracy of the base model before fine-tuning, corroborating the reference paper's findings.

## 7) Model and Trainer Setup
* Initializes the `EncoderDecoderModel` using the `cahya/bert2bert-indonesian-summarization` checkpoint.
* Configures sequence-to-sequence generation parameters (e.g., beam search, length penalty, early stopping, and n-gram repetition constraints).
* Sets up the `Seq2SeqTrainer` with 10 epochs, a data collator, and a custom `compute_metrics` function to calculate ROUGE scores during training.

## 8) Pre-fine-tuning Baseline Evaluation
* Selects a reproducible subset of 200 samples from the test set.
* Generates summaries using the base (pre-fine-tuned) model.
* Computes baseline ROUGE scores to establish a benchmark for quantifying the benefits of the fine-tuning process.

## 9) Fine-tuning
* Executes the fine-tuning process using the `trainer.train()` method on the 17,500-sample Indonesian training dataset.
* Saves the fully fine-tuned model and tokenizer back to Google Drive for persistence and future inference.

## 10) Post-fine-tuning Evaluation
* Evaluates the model using two approaches:
  1. Official `trainer.evaluate()` on the full 5,000-sample tokenized test split.
  2. Direct inference using `model.generate()` on the exact same 200-sample baseline subset used in Section 8 to allow a direct "before-and-after" comparison.

## 11) Comparison Table & Qualitative Demo
* **Quantitative:** Builds a Pandas DataFrame comparing the ROUGE scores of the pre-fine-tuned and fine-tuned models against the published baselines from the AACL 2020 paper.
* **Qualitative:** Displays 5 sample articles side-by-side, showcasing the original text, the human reference summary, the pre-fine-tuning output, and the post-fine-tuning output for visual inspection.